### Install Libraries

In [ ]:
# !!pip install torch transformers datasets peft trl accelerate bitsandbytes

### Import Libraries

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer, SFTConfig

In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

### Load Model and Tokenizer

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

### Load dataset and preprocessing it

In [ ]:
# Dataset needs a text column OR prompt+completion columns
# Common format: {"prompt": "...", "completion": "..."}
# TRL's SFTTrainer can also handle chat templates automatically
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
# Set chat template for base model
tokenizer.chat_template = (
    "{{ bos_token }}"                        # <s> at the very start
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "[INST] {{ message['content'] }} [/INST]"
    "{% elif message['role'] == 'assistant' %}"
    "{{ message['content'] }}{{ eos_token }}" # </s> after every assistant turn
    "{% endif %}"
    "{% endfor %}"
)

# ── Step 2: Load and format dataset ──
# raw_dataset = load_dataset(
#     "HuggingFaceH4/ultrachat_200k",
#     split="train_sft"
# )

def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return example

dataset = dataset.map(
    apply_chat_template,
    num_proc=1,
    desc="Applying chat template",
    # remove all other columns — leave only "text"
    # this prevents TRL from seeing "prompt" column and going into assistant_only_loss path
    remove_columns=dataset.column_names,
)

# train_dataset = dataset.select(range(200))
# Verify — should ONLY have "text" column now
print("Columns:", dataset.column_names)   # → ['text']
print("Sample:\n", dataset[0]["text"][:200])

### Setting up SFT config and Trainer

In [ ]:
training_args = SFTConfig(
    output_dir="./sft-full",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,           # Standard SFT LR
    # max_seq_length=2048,
    bf16=True,
    logging_steps=10,
    save_steps=200,
    dataset_text_field="prompt",  # column to train on
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

### To start training run command below

In [ ]:
trainer.train()

### Saved trained model

In [ ]:
# ============================================================
# SAVE — saves config.json, model weights, tokenizer files
# ============================================================
trainer.save_model("./sft-full-saved")
tokenizer.save_pretrained("./sft-full-saved")
# output dir has: config.json, pytorch_model.bin (or shards), tokenizer.json, etc.

### Below Step is to Load trained model and predict based on query

In [ ]:
# ============================================================
# LOAD
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "./sft-full-saved",
    torch_dtype="auto",
    device_map="auto",      # auto-places layers across available GPUs/CPU
)
tokenizer = AutoTokenizer.from_pretrained("./sft-full-saved")

In [ ]:
# ============================================================
# PREDICT — identical interface regardless of load option
# ============================================================
import torch

def predict(prompt: str, max_new_tokens: int = 200) -> str:
    # Format using the same template used during training
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,   # adds [/INST] at end — model generates after this
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )

    # Fix: explicitly send to cuda instead of model.device
    # model.device fails when device_map="auto" splits layers across devices
    input_ids = inputs["input_ids"].to("cuda")
    attention_mask = inputs["attention_mask"].to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Slice off the prompt tokens — only decode the newly generated part
    generated_tokens = outputs[0][input_ids.shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

# ── Test it ──────────────────────────────────────────────────────────────────
print(predict("What is the difference between L1 and L2 regularization?"))
print("---")
